# Exploratory Analysis of House Prices

**Mid-semester project | Ames, Iowa | 2006–2010**

This notebook explores 1,460 recorded home sales and asks how price varies with overall quality, living area, garage capacity and neighborhood. It focuses on descriptive analysis. Correlation does not establish cause, and the historical data should not be used as a current-market estimate.

The companion file `ames_housing_train.csv` is included beside this notebook. The source is the Kaggle *House Prices: Advanced Regression Techniques* training split, mirrored at Hugging Face. See the final section for references.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="crest")
DATA_FILE = Path("ames_housing_train.csv")
df = pd.read_csv(DATA_FILE)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head()

## 1. Understand the data

Each row represents a home sale. `SalePrice` is the target recorded in the source dataset. The other columns describe sale conditions, location, quality, size and amenities. `Id` is a row identifier, so it is not treated as a housing characteristic.

In [ ]:
df.info()

In [ ]:
summary = df["SalePrice"].describe(percentiles=[.25, .5, .75]).round(2)
print(summary)
print("Target skewness:", round(df["SalePrice"].skew(), 3))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Ids:", df["Id"].duplicated().sum())

## 2. Missing values

Blanks in optional-amenity fields can mean the home does not have that feature. Inspect the dataset description before choosing an imputation rule.

In [ ]:
missing = (df.isna().sum().rename("missing")
           .to_frame().assign(percent=lambda x: 100*x["missing"]/len(df))
           .query("missing > 0").sort_values("missing", ascending=False))
missing.head(15)

In [ ]:
top_missing = missing.head(8).sort_values("percent")
ax = top_missing["percent"].plot(kind="barh", figsize=(8, 4), color="#D98145")
ax.set(title="Columns with the most missing values", xlabel="Rows missing (%)", ylabel="")
plt.tight_layout()
plt.show()

## 3. Sale-price distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["SalePrice"], bins=35, kde=True, ax=axes[0], color="#2A8C82")
axes[0].set(title="Sale price distribution", xlabel="Sale price ($)")
sns.boxplot(x=df["SalePrice"], ax=axes[1], color="#9CC9C4")
axes[1].set(title="Price spread", xlabel="Sale price ($)")
plt.tight_layout()
plt.show()

The mean is about **$180,921** and the median is **$163,000**. The mean sitting above the median, along with skewness of about **1.88**, shows the long right tail.

## 4. Numeric relationships with SalePrice

Pearson correlation measures the strength and direction of a linear relationship between two numeric columns. It does not control for other attributes and does not imply causality.

In [ ]:
numeric = df.select_dtypes(include="number").drop(columns=["Id", "SalePrice"])
corr = numeric.corrwith(df["SalePrice"]).sort_values(ascending=False)
print(corr.head(10).round(3))

In [ ]:
top_corr = corr.head(8).sort_values()
ax = top_corr.plot(kind="barh", figsize=(8, 4), color="#2A8C82")
ax.set(title="Strongest numeric correlations with sale price", xlabel="Pearson r", ylabel="")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", alpha=.55, s=28, ax=ax, color="#2A8C82")
ax.set(title="Above-ground living area and sale price", xlabel="Above-ground living area (sq ft)", ylabel="Sale price ($)")
plt.tight_layout()
plt.show()

`OverallQual` has the strongest numeric correlation with price (**r = 0.791**), followed by `GrLivArea` (**r = 0.709**). A small number of large homes have unusually low prices relative to their size, so the scatterplot also shows why outliers deserve inspection rather than automatic removal.

## 5. Price by overall quality

In [ ]:
quality_summary = (df.groupby("OverallQual")["SalePrice"]
                   .agg(count="size", median="median", mean="mean"))
display(quality_summary.round(0))
ax = quality_summary["median"].plot(kind="bar", figsize=(8, 4), color="#2A8C82")
ax.set(title="Median sale price by overall quality", xlabel="Overall quality score", ylabel="Median sale price ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Median price increases from **$50,150** at quality 1 to **$432,390** at quality 10. The extreme levels contain few homes (2 at level 1 and 18 at level 10), so those medians are less stable than the middle levels.

## 6. Neighborhood differences

To avoid over-interpreting very small groups, the comparison keeps neighborhoods with at least 20 recorded sales.

In [ ]:
neighborhood_summary = (df.groupby("Neighborhood")["SalePrice"]
                        .agg(count="size", median="median", mean="mean")
                        .query("count >= 20")
                        .sort_values("median"))
display(neighborhood_summary.round(0))
ax = neighborhood_summary["median"].plot(kind="barh", figsize=(8, 7), color="#2A8C82")
ax.set(title="Median sale price by neighborhood (n >= 20)", xlabel="Median sale price ($)", ylabel="")
plt.tight_layout()
plt.show()

Among neighborhoods with at least 20 sales, median prices range from **$103,000** in IDOTRR to **$315,000** in NridgHt. This describes different mixes of sold homes; it does not isolate a causal location premium.

## 7. Summary and limitations

- The price distribution is right-skewed, so the median is a useful measure of a typical sale.
- Overall quality and living area show the largest pairwise numeric correlations with price.
- Median prices rise across quality groups and differ across neighborhoods.
- Missingness clusters in optional-amenity fields. Check the codebook before treating blanks as unknown values.

**Limitations:** one city, sales from 2006–2010, unequal group sizes, and observational associations. This notebook is EDA, not a current valuation tool or a predictive model.

## References

1. Kaggle, [House Prices: Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data), training split. Dataset mirror: [Hugging Face, t22000t/house-prices-tabular](https://huggingface.co/datasets/t22000t/house-prices-tabular).
2. De Cock, D. (2011). *Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project*. Journal of Statistics Education, 19(3). https://doi.org/10.1080/10691898.2011.11889627